In [1]:
import re
from typing import Dict, List, Tuple
import spacy

In [2]:
nlp = spacy.load(
    "en_core_web_trf",
    disable=["ner", "textcat", "lemmatizer", "tagger"],
)

def clean_sentence(text: str) -> str:
    # Keep letters, digits, whitespace, apostrophes and hyphens.
    text = re.sub(r"[^\w\s'-]", "", text)
    text = " ".join(text.split())
    return text


def remove_single_dependency(sentence: str, dep_to_remove: str) -> Dict:
    """
    Removes one selected spaCy dependency construction from a single sentence.

    Parameters
    ----------
    sentence : str
        Input sentence.
    dep_to_remove : str
        Dependency label to remove, e.g. "appos", "acl", "advcl", "relcl".

    Returns
    -------
    Dict
        {
            "original": original sentence,
            "dependency": selected dependency label,
            "removed_fragments": removed text spans,
            "simplified": simplified sentence
        }
    """
    doc = nlp(sentence)

    to_remove = set()
    removed_fragments = []

    for token in doc:
        if token.dep_ == dep_to_remove:
            subtree_tokens = list(token.subtree)
            subtree_ids = {t.i for t in subtree_tokens}

            to_remove.update(subtree_ids)

            removed_text = "".join(t.text_with_ws for t in subtree_tokens).strip()
            removed_fragments.append(removed_text)

    pruned = "".join(
        token.text_with_ws
        for token in doc
        if token.i not in to_remove
    )

    simplified = clean_sentence(pruned)

    return {
        "original": sentence,
        "dependency": dep_to_remove,
        "removed_fragments": removed_fragments,
        "simplified": simplified,
    }

In [8]:
examples = [
    ("The book that I read yesterday was fascinating.", "relcl"),
    ("My friend, a well-known scientist, published a paper.", "appos"),
    # ("The decision approved by the committee was controversial.", "acl" ),
    ("The student selected for the scholarship gave a speech.", "acl"),
    ("The president, speaking in Paris, announced sanctions while addressing the media.", "advcl"),
]   
expected_result = "The decision was controversial"

for sentence, dep in examples:
    result = remove_single_dependency(sentence, dep)

    print("Original:   ", result["original"])
    print("Dependency: ", result["dependency"])
    print("Removed:    ", result["removed_fragments"])
    print("Simplified: ", result["simplified"])
    print()

Original:    The book that I read yesterday was fascinating.
Dependency:  relcl
Removed:     ['that I read yesterday']
Simplified:  The book was fascinating

Original:    My friend, a well-known scientist, published a paper.
Dependency:  appos
Removed:     ['a well-known scientist']
Simplified:  My friend published a paper

Original:    The student selected for the scholarship gave a speech.
Dependency:  acl
Removed:     ['selected for the scholarship']
Simplified:  The student gave a speech

Original:    The president, speaking in Paris, announced sanctions while addressing the media.
Dependency:  advcl
Removed:     ['speaking in Paris', 'while addressing the media']
Simplified:  The president announced sanctions

